# 01 – Data Cleaning
**Hotel Booking Analytics – Capstone Project** · Junaidi Abdul Rahim (Arjun)

**Goal:** turn the raw booking files into one clean, checked table that Excel, Power BI and the analysis can all use.

**Run order:** `01_Clean.ipynb` → `02_Analyse.ipynb` → `03_Charts.ipynb`

**Folder set-up** (run the notebooks from the `notebooks/` folder):
```
data/raw/         Bookings.csv, Hotels.csv, Market Segments.csv, Customer Types.csv, Reservation Status.csv
data/processed/   outputs are saved here (created automatically)
images/charts/    chart images are saved here
```

## Step 1 – Import libraries and load the raw files

In [1]:
import os                      # to create folders
import pandas as pd            # tables (DataFrames)
import numpy as np             # maths helpers, e.g. np.where
import pycountry               # turns country codes (PRT) into names (Portugal)

os.makedirs('../data/processed', exist_ok=True)    # make the output folder if it does not exist
U = '../data/raw/'                          # folder that holds the raw CSV files

b   = pd.read_csv(U + 'Bookings.csv')             # main table: one row per booking
hot = pd.read_csv(U + 'Hotels.csv')               # lookup: Hotel_ID -> hotel name
ms  = pd.read_csv(U + 'Market Segments.csv')      # lookup: segment + distribution channel
ct  = pd.read_csv(U + 'Customer Types.csv')       # lookup: customer type, deposit, meal
rs  = pd.read_csv(U + 'Reservation Status.csv')   # lookup: Check-Out / Canceled / No-Show

print('Bookings:', b.shape)    # rows and columns of the raw bookings
b.head()                       # show the first 5 rows

Bookings: (119390, 28)


,Booking_ID,Hotel_ID,Market_Segment_ID,Customer_Type_ID,Reservation_ID,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,...,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,reservation_status_date
0,1,1,1,1,1,0,342,2015,July,27,...,0,0,3,NaN,NaN,0,0.0,0,0,2015-07-01
1,2,1,1,1,1,0,737,2015,July,27,...,0,0,4,NaN,NaN,0,0.0,0,0,2015-07-01
2,3,1,1,1,1,0,7,2015,July,27,...,0,0,0,NaN,NaN,0,75.0,0,0,2015-07-02
3,4,1,2,1,1,0,13,2015,July,27,...,0,0,0,304.0,NaN,0,75.0,0,0,2015-07-02
4,5,1,3,1,1,0,14,2015,July,27,...,0,0,0,240.0,NaN,0,98.0,0,1,2015-07-03


## Step 2 – A cleaning log
Every cleaning step is written into a log, so each change can be explained and checked later (it is also shown in the Excel workbook).

In [2]:
log = []                                   # empty list that will hold one entry per step

def L(step, desc, rows):
    # add one step to the log: step name, what was done, how many rows it affected
    log.append({'Step': len(log) + 1, 'Action': step, 'Detail': desc, 'Rows affected': int(rows)})

n0 = len(b)                                # remember the starting number of rows
L('Load', 'Bookings table loaded (raw rows)', n0)

## Step 3 – Check the keys (do the IDs match the lookup tables?)

In [3]:
bad_keys = ((~b.Hotel_ID.isin(hot.Hotel_ID)).sum()                     # hotel IDs not in Hotels
          + (~b.Market_Segment_ID.isin(ms.Market_Segment_ID)).sum()    # segment IDs not in Market Segments
          + (~b.Customer_Type_ID.isin(ct.Customer_Type_ID)).sum()      # customer IDs not in Customer Types
          + (~b.Reservation_ID.isin(rs.Reservation_ID)).sum())         # status IDs not in Reservation Status
L('Check keys', 'All foreign keys match lookup tables (Hotel, Segment, Customer Type, Reservation)', bad_keys)
print('Keys that do not match:', bad_keys)     # 0 means every booking links correctly

Keys that do not match: 0


## Step 4 – Fill missing values

In [4]:
print(b[['children', 'country', 'agent', 'company']].isna().sum())   # how many blanks before filling

L('Missing values', 'children blank -> 0', b.children.isna().sum())
b['children'] = b.children.fillna(0).astype(int)          # a blank number of children means none

L('Missing values', 'country blank -> "UNK" (Unknown)', b.country.isna().sum())
b['country'] = b.country.fillna('UNK')                    # unknown country gets the code UNK

L('Missing values', 'agent blank -> 0 = "No agent" (booked without travel agent)', b.agent.isna().sum())
L('Missing values', 'company blank -> 0 = "No company" (not a corporate booking)', b.company.isna().sum())
b['agent'] = b.agent.fillna(0).astype(int)                # 0 = no travel agent (booked directly)
b['company'] = b.company.fillna(0).astype(int)            # 0 = no company (not a corporate booking)

children         4
country        488
agent        16340
company     112593
dtype: int64


## Step 5 – Remove invalid rows (182 in total)

In [5]:
m = (b.adults + b.children + b.babies) == 0                # bookings with no guests at all
L('Remove invalid', 'Bookings with 0 guests (adults+children+babies=0)', m.sum())
b = b[~m]                                                  # keep everything that is NOT in m

m = b.adr < 0                                              # a negative room price is impossible
L('Remove invalid', 'Negative ADR', m.sum())
b = b[~m]

m = b.adr > 1000                                           # one booking priced at 5,400 euros: a typing error
L('Remove outlier', 'ADR above 1,000 (single 5,400 entry error)', m.sum())
b = b[~m]

print('Rows removed so far:', n0 - len(b))

Rows removed so far: 182


## Step 6 – Check duplicates and consistency
Rows that look identical are **kept**: there is no guest ID, so they are most likely separate rooms in the same group booking.

In [6]:
dups = b.drop(columns='Booking_ID').duplicated().sum()    # identical rows if we ignore the booking ID
L('Check duplicates', 'Identical rows (excluding Booking_ID) kept: there is no guest ID, so they are most likely separate rooms in the same group booking.', dups)

mismatch = ((b.Reservation_ID > 1).astype(int) != b.is_canceled).sum()   # status 2/3 = Canceled/No-Show must equal is_canceled = 1
L('Check consistency', 'is_canceled matches reservation status (1 = Canceled/No-Show)', mismatch)
print('Look-alike rows kept:', dups, '| status mismatches:', mismatch)

Look-alike rows kept: 32995 | status mismatches: 0


## Step 7 – Feature engineering (new columns)

In [7]:
months = ['January', 'February', 'March', 'April', 'May', 'June', 'July',
          'August', 'September', 'October', 'November', 'December']
mon = {m: i for i, m in enumerate(months, 1)}                       # {'January': 1, ..., 'December': 12}
b['arrival_month_num'] = b.arrival_date_month.map(mon)              # month name -> month number

b['arrival_date'] = pd.to_datetime(dict(year=b.arrival_date_year,  # build a real date from year, month, day
                                        month=b.arrival_month_num,
                                        day=b.arrival_date_day_of_month))
b['reservation_status_date'] = pd.to_datetime(b.reservation_status_date)    # text -> date
b['booking_date'] = b.arrival_date - pd.to_timedelta(b.lead_time, unit='D') # arrival date minus lead time

b['total_nights'] = b.stays_in_weekend_nights + b.stays_in_week_nights     # length of stay
b['total_guests'] = b.adults + b.children + b.babies                       # party size
b['has_kids'] = ((b.children + b.babies) > 0).astype(int)                  # 1 = family with children

b['booking_value'] = (b.adr * b.total_nights).round(2)                                # price per night x nights
b['realised_revenue'] = np.where(b.is_canceled == 0, b.booking_value, 0).round(2)    # money from guests who stayed
b['lost_revenue'] = np.where(b.is_canceled == 1, b.booking_value, 0).round(2)        # value of cancelled bookings

b['lead_time_band'] = pd.cut(b.lead_time, [-1, 7, 30, 90, 180, 365, 1000],           # group lead time into bands
                             labels=['0-7 days', '8-30 days', '31-90 days', '91-180 days',
                                     '181-365 days', '365+ days']).astype(str)
b['stay_length_band'] = pd.cut(b.total_nights, [-1, 0, 3, 7, 14, 100],               # group stay length into bands
                               labels=['Day use (0)', '1-3 nights', '4-7 nights',
                                       '8-14 nights', '15+ nights']).astype(str)

def season(x):
    # month number -> season name
    if x in (6, 7, 8): return 'Summer (Jun-Aug)'
    if x in (9, 10, 11): return 'Autumn (Sep-Nov)'
    if x in (12, 1, 2): return 'Winter (Dec-Feb)'
    return 'Spring (Mar-May)'
b['season'] = b.arrival_month_num.map(season)

b['arrival_weekday'] = b.arrival_date.dt.day_name()                  # Monday, Tuesday, ...
b['has_agent'] = (b.agent > 0).astype(int)                           # 1 = booked through a travel agent
b['has_company'] = (b.company > 0).astype(int)                       # 1 = company booking
b['has_previous_cancel'] = (b.previous_cancellations > 0).astype(int)  # 1 = guest cancelled before

L('Feature engineering', 'Added arrival_date, booking_date, total_nights, total_guests, has_kids, booking_value, realised_revenue, lost_revenue, lead_time_band, stay_length_band, season, weekday, has_agent, has_company flags', len(b))
b[['arrival_date', 'booking_date', 'total_nights', 'booking_value', 'realised_revenue', 'lost_revenue', 'lead_time_band', 'season']].head()

,arrival_date,booking_date,total_nights,booking_value,realised_revenue,lost_revenue,lead_time_band,season
0,2015-07-01,2014-07-24,0,0.0,0.0,0.0,181-365 days,Summer (Jun-Aug)
1,2015-07-01,2013-06-24,0,0.0,0.0,0.0,365+ days,Summer (Jun-Aug)
2,2015-07-01,2015-06-24,1,75.0,75.0,0.0,0-7 days,Summer (Jun-Aug)
3,2015-07-01,2015-06-18,1,75.0,75.0,0.0,8-30 days,Summer (Jun-Aug)
4,2015-07-01,2015-06-17,2,196.0,196.0,0.0,8-30 days,Summer (Jun-Aug)


## Step 8 – Country codes to country names

In [8]:
def cname(c):
    # turn a 3-letter country code into a name
    if c == 'UNK': return 'Unknown'            # our own code for blanks
    if c == 'CN':  return 'China'              # the data uses CN instead of CHN
    if c == 'TMP': return 'East Timor'         # old code for Timor-Leste
    r = pycountry.countries.get(alpha_3=c)     # look up the official code
    return r.name if r else c                  # if not found, keep the code

countries = pd.DataFrame({'country': sorted(b.country.unique())})   # one row per country code
countries['country_name'] = countries.country.map(cname)            # add the name
b = b.merge(countries, on='country', how='left')                    # attach the name to every booking
print('Countries:', len(countries))

Countries: 178


## Step 9 – Save the clean data (star schema for Power BI) and the cleaning log

In [9]:
L('Final', 'Clean fact table rows', len(b))
L('Final', 'Rows removed in total', n0 - len(b))

b.to_pickle('../data/processed/bookings.pkl')                                     # fast copy for the next notebook
b.drop(columns=['country_name']).to_csv('../data/processed/Fact_Bookings.csv',    # fact table (country name lives in Dim_Country)
                                        index=False, date_format='%Y-%m-%d')
hot.to_csv('../data/processed/Dim_Hotel.csv', index=False)                        # dimension tables
ms.to_csv('../data/processed/Dim_Market_Segment.csv', index=False)
ct.to_csv('../data/processed/Dim_Customer_Type.csv', index=False)
rs.to_csv('../data/processed/Dim_Reservation_Status.csv', index=False)
countries.to_csv('../data/processed/Dim_Country.csv', index=False)

d = pd.DataFrame({'Date': pd.date_range(b.booking_date.min().replace(day=1), '2018-12-31')})   # calendar table
d['Year'] = d.Date.dt.year
d['MonthNum'] = d.Date.dt.month
d['Month'] = d.Date.dt.strftime('%b')
d['YearMonth'] = d.Date.dt.strftime('%Y-%m')
d['Quarter'] = 'Q' + d.Date.dt.quarter.astype(str)
d['Weekday'] = d.Date.dt.day_name()
d['IsWeekend'] = d.Date.dt.weekday.isin([5, 6]).astype(int)          # Saturday or Sunday
d.to_csv('../data/processed/Dim_Date.csv', index=False, date_format='%Y-%m-%d')

log_df = pd.DataFrame(log)                                            # the full cleaning log
log_df.to_csv('../data/processed/cleaning_log.csv', index=False)
log_df

,Step,Action,Detail,Rows affected
0,1,Load,Bookings table loaded (raw rows),119390
1,2,Check keys,"All foreign keys match lookup tables (Hotel, S...",0
2,3,Missing values,children blank -> 0,4
3,4,Missing values,"country blank -> ""UNK"" (Unknown)",488
4,5,Missing values,"agent blank -> 0 = ""No agent"" (booked without ...",16340
5,6,Missing values,"company blank -> 0 = ""No company"" (not a corpo...",112593
6,7,Remove invalid,Bookings with 0 guests (adults+children+babies=0),180
7,8,Remove invalid,Negative ADR,1
8,9,Remove outlier,"ADR above 1,000 (single 5,400 entry error)",1
9,10,Check duplicates,Identical rows (excluding Booking_ID) kept: th...,32995


### Summary
- 119,390 raw bookings → **119,208 clean bookings** (182 removed: 180 with 0 guests, 1 negative price, 1 price of €5,400).
- 32,995 look-alike rows **kept** (group rooms, no guest ID).
- 17 new columns added, including booking value, realised and lost revenue.
- Next: run **02_Analyse.ipynb**.